In [2]:
import torch
import xarray as xr
import numpy as np
import sys
sys.path.append('/home/sc.uni-leipzig.de/fl53wumy/llaae_new/DistributionalPrincipalAutoencoder')
import utils as ut
import json

# Transform torch to xarray

In [2]:
def torch_to_dataarray(x_tensor, coords_ds, lat_dim=32, lon_dim=32, name="variable"):
    """
    Convert a flattened 2D torch tensor to a 3D xarray.DataArray (lat, lon, time).

    Parameters:
    ----------
    x_tensor : torch.Tensor
        A 2D tensor of shape (time_steps, lat_dim * lon_dim), or a 1D tensor to be reshaped.
    lat_dim : int
        Number of latitude points.
    lon_dim : int
        Number of longitude points.
    coords_ds : xarray.Dataset
        Dataset containing 'lat', 'lon', and 'time' coordinates to assign.
    name : str, optional
        Name of the variable in the DataArray.

    Returns:
    -------
    xarray.DataArray
        The reshaped and labeled data as an xarray.DataArray.
    """
    # Step 1: Convert to NumPy
    # data_np = x_tensor.detach().cpu().numpy()

    # Step 1: Convert to NumPy
    if isinstance(x_tensor, np.ndarray):
        # x is a NumPy array
        print("input array is numpy array")
        data_np = x_tensor
    elif isinstance(x_tensor, torch.Tensor):
        print("input array is torch tensor")
        data_np = x_tensor.detach().cpu().numpy()

    # Step 2: Determine time_steps and reshape
    print("data_np dimensions:", data_np.ndim)
    #if data_np.ndim == 2:
    #    time_steps = 1
    #elif data_np.ndim > 2:
    time_steps = data_np.shape[0]
    
    data_np = data_np.reshape(time_steps, lat_dim, lon_dim)

    # Step 3: Transpose to (lat, lon, time)
    data_np = data_np.transpose(1, 2, 0)

    # Step 4: Create the DataArray
    da = xr.DataArray(
        data_np,
        dims=("lat", "lon", "time"),
        coords={
            "lat": coords_ds.lat,
            "lon": coords_ds.lon,
            "time": np.arange(time_steps)
        },
        name=name
    )

    return da

In [3]:
# load reference dataset
#ds_ref = xr.open_dataset("/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/dpa_input_data/v1_until16102025/europe_10percent_masked_stacked_TREFHT_JJA.nc")
#ds_ref

settings_file_path =  "/home/sc.uni-leipzig.de/fl53wumy/llaae_new/DistributionalPrincipalAutoencoder/joint_training/v4_dpa_train_settings.json" #"../joint_training/v2_dpa_train_settings.json"
    
with open(settings_file_path, 'r') as file:
        settings = json.load(file)

# Train data
ds_ref = xr.open_dataset(settings['dataset_trefht'])
ds_ref

<xarray.Dataset> Size: 2GB
Dimensions:  (lat: 32, lon: 32, time: 476900)
Coordinates:
  * lon      (lon) float64 256B -11.25 -10.0 -8.75 -7.5 ... 25.0 26.25 27.5
  * lat      (lat) float64 256B 34.4 35.34 36.28 37.23 ... 61.73 62.67 63.61
  * time     (time) object 4MB 1850-06-02 00:00:00 ... 2100-08-31 00:00:00
Data variables:
    TREFHT   (lat, lon, time) float32 2GB ...

In [5]:
ds_torch_sample = ut.data_to_torch(ds_ref, "TREFHT")
print(ds_torch_sample.shape)
_, mask_trefht = ut.remove_nan_columns(ds_torch_sample)

torch.Size([476900, 1024])
torch.Size([476900, 1024])


In [6]:
# load StoNet predicted torch ensemble
torch_ensemble_fact = torch.load("/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/StoNet/v4_spatial/_6_50_5_bs128_bnisFalse_30_epochs_trained/factual_30_epochs_trained_engressor_predicted_ensemble.pt", map_location="cpu")
torch_ensemble_cf = torch.load("/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/StoNet/v4_spatial/_6_50_5_bs128_bnisFalse_30_epochs_trained/cf_30_epochs_trained_engressor_predicted_ensemble.pt", map_location="cpu")

torch_ensemble_fact.shape


# analogue_ensemble_member","time","lat","lon"

torch.Size([14307, 648, 100])

In [7]:
import matplotlib.pyplot as plt
torch_ensemble_fact_restored = torch.full((100, 14307,32, 32), float("nan"))
for i in range(100):
    print(i)
    print(torch_ensemble_fact[:,:,i].shape)
    gen1_te_from_restored_fact = ut.restore_nan_columns(torch_ensemble_fact[:,:,i], mask_trefht)
    #print(gen1_te_from_restored.shape)
    reconstructed_da_fact = torch_to_dataarray(gen1_te_from_restored_fact, coords_ds=ds_ref, name="Temperature")
    #print(reconstructed_da_fact.shape)
    #reconstructed_da.isel(time=5).plot()
    #plt.show()
    torch_ensemble_fact_restored[i,:,:,:] = torch.from_numpy(reconstructed_da_fact.values.transpose(2,0,1))

torch_ensemble_cf_restored = torch.full((100, 14307,32, 32), float("nan"))
for i in range(100):
    print(i)
    print(torch_ensemble_cf[:,:,i].shape)
    gen1_te_from_restored_cf = ut.restore_nan_columns(torch_ensemble_cf[:,:,i], mask_trefht)
    #print(gen1_te_from_restored_cf.shape)
    reconstructed_da_cf = torch_to_dataarray(gen1_te_from_restored_cf, coords_ds=ds_ref, name="Temperature")
    #print(reconstructed_da_cf.shape)
    #reconstructed_da.isel(time=5).plot()
    #plt.show()
    torch_ensemble_cf_restored[i,:,:,:] = torch.from_numpy(reconstructed_da_cf.values.transpose(2,0,1))

0
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
1
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
2
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
3
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
4
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
5
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
6
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
7
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
8
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
9
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
10
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
11
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions: 2
12
torch.Size([14307, 648])
input array is torch tensor
data_np dimensions

In [9]:
# create an xarray from it
# convert torch -> numpy (handles GPU + autograd safely)
data_fact = torch_ensemble_fact_restored.detach().cpu().numpy()
ensemble_fact_xr = xr.DataArray(
    data_fact,
    dims=("ensemble_member", "time", "lat", "lon"),
    coords={
        "ensemble_member": np.arange(100),
        "time": ds_ref.time.isel(time=slice(0,3*4769)),
        "lat": ds_ref.lat,
        "lon": ds_ref.lon,
    },
    name = "TREFHT"
)

data_cf = torch_ensemble_cf_restored.detach().cpu().numpy()
ensemble_cf_xr = xr.DataArray(
    data_cf,
    dims=("ensemble_member", "time", "lat", "lon"),
    coords={
        "ensemble_member": np.arange(100),
        "time": ds_ref.time.isel(time=slice(0,3*4769)),
        "lat": ds_ref.lat,
        "lon": ds_ref.lon,
    },
    name = "TREFHT"
)
ensemble_cf_xr

<xarray.DataArray 'TREFHT' (ensemble_member: 100, time: 14307, lat: 32, lon: 32)> Size: 6GB
array([[[[        nan,         nan,         nan, ...,         nan,
                  nan,         nan],
         [        nan,         nan,         nan, ..., -0.7121208 ,
                  nan,         nan],
         [        nan,         nan,         nan, ...,         nan,
                  nan,         nan],
         ...,
         [        nan,         nan,         nan, ...,  2.68578   ,
           3.0057912 ,  3.135509  ],
         [        nan,         nan,         nan, ...,  3.1021936 ,
           3.4283996 ,  2.6581185 ],
         [        nan,         nan,         nan, ...,  3.449655  ,
           3.4135823 ,  4.1622505 ]],

        [[        nan,         nan,         nan, ...,         nan,
                  nan,         nan],
         [        nan,         nan,         nan, ...,  1.097324  ,
                  nan,         nan],
         [        nan,         nan,         nan, ...,         nan,
                  nan,         nan],
...
          -0.47036344, -0.01469922],
         [        nan,         nan,         nan, ..., -0.25921908,
          -0.3857684 , -0.2763853 ],
         [        nan,         nan,         nan, ..., -0.37966013,
          -0.15652144, -0.11477232]],

        [[        nan,         nan,         nan, ...,         nan,
                  nan,         nan],
         [        nan,         nan,         nan, ..., -0.15297705,
                  nan,         nan],
         [        nan,         nan,         nan, ...,         nan,
                  nan,         nan],
         ...,
         [        nan,         nan,         nan, ..., -2.6917145 ,
          -3.1805537 , -1.9928595 ],
         [        nan,         nan,         nan, ..., -2.511869  ,
          -2.2748713 , -2.5135794 ],
         [        nan,         nan,         nan, ..., -2.7943194 ,
          -2.7468464 , -2.9988995 ]]]],
      shape=(100, 14307, 32, 32), dtype=float32)
Coordinates:
  * ensemble_member  (ensemble_member) int64 800B 0 1 2 3 4 5 ... 95 96 97 98 99
  * time             (time) object 114kB 1850-06-02 00:00:00 ... 2100-08-31 0...
  * lat              (lat) float64 256B 34.4 35.34 36.28 ... 61.73 62.67 63.61
  * lon              (lon) float64 256B -11.25 -10.0 -8.75 ... 25.0 26.25 27.5

In [5]:
# load dpa predicted ensemble for comparison
#ds_dpa_ref = xr.open_dataset("/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/dpa_output/v4_model/_50_6_50_5_1001_20_2_50_encoderislearnable_lambda0.5_bs128_bnisTrue/dpa_ensemble_after_20_epochs/eth_ensemble_after_20_epochs/ETH_gen_dpa_ens_20_dataset_restored.nc")
ds_dpa_ref = xr.open_dataset("/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/dpa_output/v4_model/_50_6_50_5_1001_20_2_50_encoderislearnable_lambda0.5_bs128_bnisTrue/dpa_ensemble_after_20_epochs/eth_ensemble_after_20_epochs/raw_ETH_gen_dpa_ens_20_dataset.nc")
ds_dpa_ref

<xarray.Dataset> Size: 4GB
Dimensions:          (ensemble_member: 100, time: 14307, lat_x_lon: 648)
Coordinates:
  * ensemble_member  (ensemble_member) int64 800B 1 2 3 4 5 ... 96 97 98 99 100
  * time             (time) object 114kB 1850-06-02 00:00:00 ... 2100-08-31 0...
  * lat_x_lon        (lat_x_lon) int64 5kB 0 1 2 3 4 5 ... 643 644 645 646 647
Data variables:
    TREFHT           (ensemble_member, time, lat_x_lon) float32 4GB ...

In [10]:
# plot sample from xarray
#ensemble_xr.isel(ensemble_member=0, time=0).plot()
# save the xarray

#ensemble_fact_xr.to_netcdf("/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/StoNet/v4_spatial/_6_50_5_bs128_bnisFalse_30_epochs_trained/restored_StoNet_predicted_fact_ensembles_ETH_predictors_standardized_training_and_inference_30_epochs_trained_v4_data.nc")
#ensemble_cf_xr.to_netcdf("/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/StoNet/v4_spatial/_6_50_5_bs128_bnisFalse_30_epochs_trained/restored_StoNet_predicted_cf_ensembles_ETH_predictors_standardized_training_and_inference_30_epochs_trained_v4_data.nc")